# Brusselator parameter atlas

This notebook is the slower companion to `assets/brusselator-parameter-atlas.svg`.

The goal is narrow and chemistry-facing: take the Brusselator beyond a single `B` sweep and ask how the exact Hopf curve `B = 1 + A^2` organizes **when oscillation appears, how large it gets, and how fast it cycles**.

That turns the repo's autocatalytic example into a small parameter-study lab instead of one pretty phase portrait.

## Exact local theory first

For

$$\dot x = A + x^2 y - (B + 1)x, \qquad \dot y = Bx - x^2 y,$$

the fixed point is

$$ (x^*, y^*) = (A, B/A). $$

At that point the Jacobian is

$$ J(x^*, y^*) = \begin{bmatrix} B - 1 & A^2 \\ -B & -A^2 \end{bmatrix}, $$

so

$$ \operatorname{tr}(J) = B - 1 - A^2, \qquad \det(J) = A^2. $$

That gives the exact Hopf boundary

$$ B = 1 + A^2. $$

Everything below that curve is locally damped. Everything above it sits on the oscillatory side of the bifurcation.

In [ ]:
from phaseportraitlab.brusselator_atlas import atlas_axis_values, build_brusselator_parameter_atlas

small_atlas = build_brusselator_parameter_atlas(
    atlas_axis_values(0.8, 1.6, 5),
    atlas_axis_values(1.2, 4.2, 9),
)
[
    (round(cell.a, 2), round(cell.b, 2), round(cell.hopf_offset, 2), round(cell.x_amplitude, 2), None if cell.period is None else round(cell.period, 2))
    for cell in small_atlas
    if -0.35 <= cell.hopf_offset <= 0.65
][:8]

## What the three atlas panels mean

1. **Top panel — exact local side of the Hopf curve.** This one is analytic. It only asks whether the fixed point attracts or repels.
2. **Middle panel — long-time `x` amplitude.** This is numerical. Below threshold it is set to zero by theory; above threshold it is estimated from the tail of the trajectory.
3. **Bottom panel — period.** This is also numerical and comes from repeated peak spacing in the tail window.

That separation matters: local stability is exact here, while orbit size and timing are empirical measurements layered on top.

In [ ]:
from phaseportraitlab.brusselator_atlas import default_brusselator_parameter_atlas
from phaseportraitlab.brusselator_sweep import brusselator_hopf_threshold

atlas = default_brusselator_parameter_atlas()
targets = [
    (0.8, brusselator_hopf_threshold(0.8) - 0.25),
    (0.8, brusselator_hopf_threshold(0.8) + 0.35),
    (1.0, brusselator_hopf_threshold(1.0) + 0.35),
    (1.5, brusselator_hopf_threshold(1.5) + 0.35),
]
anchors = [
    min(atlas, key=lambda cell: abs(cell.a - a_target) + abs(cell.b - b_target))
    for a_target, b_target in targets
]
[(round(cell.a, 2), round(cell.b, 2), round(cell.hopf_offset, 2), round(cell.x_amplitude, 2), None if cell.period is None else round(cell.period, 2)) for cell in anchors]

## Chemistry reading

In this toy kinetics model, `A` and `B` act like feed/control parameters for the reaction scheme.

The atlas makes two useful chemistry-style statements visible:

- moving upward across `B = 1 + A^2` turns a damped fixed point into a self-sustained chemical oscillator
- changing `A` does not just move the threshold; it also changes the time scale of the oscillation once the cycle appears

Numerically, the post-threshold amplitude grows smoothly on the scanned grid, which is consistent with the standard supercritical Hopf reading of the Brusselator. This notebook treats that as a measured pattern, not as a proof.

## Caveats

- The top panel is exact only because the fixed point and Jacobian are explicit.
- The amplitude and period panels are **finite-time RK4 measurements**. Very far above threshold, especially at small `A`, the excursions get large and the tail estimate becomes cruder.
- A clean Hopf boundary does not tell you every global fact about the orbit geometry. It tells you where the fixed point changes local character.

## Problems to try

1. Hold `A = 1` fixed and estimate how the measured amplitude scales with `B - 2` just above threshold. Does it look square-root-like on the sampled grid?
2. Compare two different time windows for the same high-`B`, low-`A` corner cell. How stable is the measured period?
3. Add a second chemical oscillator model and ask which parts of this atlas idea survive and which parts are specific to the Brusselator.

## References and next reads

Good follow-on references for the surrounding theory are Steven Strogatz's *Nonlinear Dynamics and Chaos* for Hopf bifurcations and the classic Brusselator literature for the reaction-kinetics origin of the model.

A natural next repo move would be to compare this local-to-global picture against another chemical oscillator or to add a second parameter map for limit-cycle size near threshold.

In [ ]:
from pathlib import Path

Path('../assets/brusselator-parameter-atlas.svg').exists(), Path('../reports/brusselator-parameter-atlas.md').exists()